# Pathway 4: ROM Concatenation

This is the **most efficient** analysis pathway. Each domain is first reduced independently via POD, and then the reduced models are concatenated.

```
Multi Solid Model  →  ROM per Domain  →  Concatenation  →  Reduced Order Model
```

This gives massive speedups for systems with many components or repeated cells.

## 1. Create Multi-Domain Geometry

In [ ]:
import os
from geometry.importers import OCCImporter
from geometry.primitives import RectangularWaveguide

a = 100e-3
L = 200e-3
maxh = 0.04

step_path = os.path.join('..', '..', 'examples', 'rwg_step_split', 'rectangular_waveguide.step')
if os.path.exists(step_path):
    geom = OCCImporter(step_path, unit='mm', auto_build=False, maxh=maxh)
    geom.add_splitting_plane_at_z(L / 2)
    geom.split()
    geom.finalize(maxh=maxh)
    print("Loaded multi-domain geometry")
else:
    geom = RectangularWaveguide(a=a, L=L, maxh=maxh)
    print("Using single-domain fallback")

## 2. Solve Per-Domain FOMs

In [ ]:
from solvers.frequency_domain import FrequencyDomainSolver

fds = FrequencyDomainSolver(geom, order=3)
fds.assemble_matrices(nmodes=1)
results = fds.solve(fmin=1.5, fmax=3.0, nsamples=30, store_snapshots=True)

print(f"Compound: {fds.is_compound}, Domains: {fds.n_domains}")
print(f"Solved at {len(fds.frequencies)} frequency points")

## 3. Reduce Each Domain Independently

Each domain's snapshot matrix is decomposed via SVD:

$$
[\mathbf{x}_1^{(d)}, \dots, \mathbf{x}_N^{(d)}] = \mathbf{U}_d \boldsymbol{\Sigma}_d \mathbf{W}_d^H
$$

The reduced matrices are:

$$
\tilde{\mathbf{K}}_d = \mathbf{V}_d^H \mathbf{K}_d \mathbf{V}_d, \quad
\tilde{\mathbf{M}}_d = \mathbf{V}_d^H \mathbf{M}_d \mathbf{V}_d
$$

In [ ]:
if fds.is_compound:
    roms = fds.foms.reduce(tol=1e-6)
    print(f"Created {len(roms)} per-domain ROMs")
    for i, rom in enumerate(roms):
        rom.print_info()
else:
    print("Single domain — see Pathway 1 for ROM reduction.")

## 4. Concatenate the ROMs

The concatenation operates on **reduced** matrices — orders of magnitude smaller than the full-order.

In [ ]:
if fds.is_compound:
    cs = roms.concatenate()
    cs.couple()
    print(f"Concatenated system external ports: {cs.ports}")
else:
    print("Skipping concatenation (single domain).")

## 5. Solve the Concatenated ROM

This solves the coupled reduced system at 1000 frequency points — typically in **under a second**.

In [ ]:
import time

if fds.is_compound:
    t0 = time.time()
    cs_results = cs.solve(fmin=1.5, fmax=3.0, nsamples=1000)
    elapsed = (time.time() - t0) * 1e3
    print(f"Concatenated ROM solve: {elapsed:.1f} ms for 1000 frequency points")
else:
    from rom.reduction import ModelOrderReduction
    rom = ModelOrderReduction(fds)
    rom.reduce(tol=1e-6)
    t0 = time.time()
    rom.solve(fmin=1.5, fmax=3.0, nsamples=1000)
    elapsed = (time.time() - t0) * 1e3
    print(f"ROM solve: {elapsed:.1f} ms for 1000 frequency points")

## 6. Plot Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

from analytical.rectangular_waveguide import RWGAnalytical

analytical = RWGAnalytical(a=a, L=L)
frequencies = np.linspace(1.5, 3.0, 200) * 1e9
S_ana = analytical.s_parameters(frequencies)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq_ghz_ana = frequencies / 1e9

if fds.is_compound:
    freq_ghz = cs.frequencies / 1e9
    s11_db = cs.get_s_db(1, 1)
    s21_db = cs.get_s_db(1, 2)
    title = 'ROM Concatenation'
else:
    freq_ghz = rom.frequencies / 1e9
    s11_db = rom.get_s_db(1, 1)
    s21_db = rom.get_s_db(1, 2)
    title = 'ROM (single domain)'

ax = axes[0]
ax.plot(freq_ghz_ana, 20*np.log10(np.abs(S_ana['S11'])+1e-15), '-', label='Analytical', lw=2)
ax.plot(freq_ghz, s11_db, '--', label=title, lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S11| (dB)')
ax.set_title('S11'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(freq_ghz_ana, 20*np.log10(np.abs(S_ana['S21'])+1e-15), '-', label='Analytical', lw=2)
ax.plot(freq_ghz, s21_db, '--', label=title, lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S21| (dB)')
ax.set_title('S21'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'{title} vs Analytical', fontsize=14)
plt.tight_layout(); plt.show()

## Performance Summary

| Method | System Size | Typical Solve Time (1000 pts) |
|--------|-------------|-------------------------------|
| FOM (Pathway 1) | ~10,000 DOFs | ~minutes |
| ROM (Pathway 1) | ~20 DOFs | ~milliseconds |
| ROM Concat (Pathway 4) | ~40 DOFs | ~milliseconds |

The key advantage of Pathway 4 is **modularity**: if one component changes, only its ROM needs recomputation.

## Summary

1. Solved each domain independently (FOM)
2. Reduced each domain via POD to a compact ROM
3. Concatenated the per-domain ROMs
4. Solved the concatenated system at 1000 points in milliseconds

This is the recommended workflow for large multi-component systems like multi-cell accelerating cavities.